In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

import scipy.stats as stats

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score

from sklearn.linear_model import LinearRegression

from sklearn.metrics import r2_score

from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import OrdinalEncoder

In [4]:
house= pd.read_csv("A:\\AIML_DEV_FOLDER\\Dataset_self\\House Price Prediction Dataset.csv")

In [5]:
house.head()

,ID,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage,Price
0,1,1360,5,4,3,1970,Downtown,Excellent,No,149919
1,2,4272,5,4,3,1958,Downtown,Excellent,No,424998
2,3,3592,2,2,3,1938,Downtown,Good,No,266746
3,4,966,4,2,2,1902,Suburban,Fair,Yes,244020
4,5,4926,1,4,2,1975,Downtown,Fair,Yes,636056


# check the data for 0 and -ve 

In [6]:
house.describe()

,ID,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Price
count,2000.000000,2000.000000,2000.000000,2000.00000,2000.000000,2000.000000,2000.000000
mean,1000.500000,2786.209500,3.003500,2.55250,1.993500,1961.446000,537676.855000
std,577.494589,1295.146799,1.424606,1.10899,0.809188,35.926695,276428.845719
min,1.000000,501.000000,1.000000,1.00000,1.000000,1900.000000,50005.000000
25%,500.750000,1653.000000,2.000000,2.00000,1.000000,1930.000000,300098.000000
50%,1000.500000,2833.000000,3.000000,3.00000,2.000000,1961.000000,539254.000000
75%,1500.250000,3887.500000,4.000000,4.00000,3.000000,1993.000000,780086.000000
max,2000.000000,4999.000000,5.000000,4.00000,3.000000,2023.000000,999656.000000


In [7]:
X = house.drop(columns=['Price'])
y = house.iloc[:,-1] # last colmun ""   --> only strenght is passed here 

In [8]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

# Note that 1st we have to convert the data in numberical data 

# Resolve this we will use Ordinal Encoding 

In [9]:
X_test.shape

(400, 9)

In [10]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder 

In [11]:

preprocessor = ColumnTransformer(
    transformers=[
        ('Location',OrdinalEncoder(categories=[['Downtown','Rural','Suburban','Urban']]),['Location']),
        ('Condition',OrdinalEncoder(categories=[['Poor', 'Fair', 'Good', 'Excellent']]),['Condition']),
        ('garage', OneHotEncoder(drop='if_binary'), ['Garage'])
    ],remainder='passthrough')

In [12]:
preprocessor.fit_transform(X_train).shape

(1600, 9)

In [13]:
preprocessor.fit_transform(X_test).shape

(400, 9)

In [14]:
X_train.head()

,ID,Area,Bedrooms,Bathrooms,Floors,YearBuilt,Location,Condition,Garage
968,969,4483,4,4,3,1933,Urban,Excellent,No
240,241,1062,3,3,1,1970,Downtown,Good,No
819,820,1422,3,4,1,1993,Urban,Good,Yes
692,693,2658,2,3,1,1972,Rural,Poor,Yes
420,421,3286,2,4,1,1981,Rural,Excellent,Yes


In [16]:
print("Train shape:", preprocessor.fit_transform(X_train).shape)
print("Test shape:", preprocessor.fit_transform(X_test).shape)

Train shape: (1600, 9)
Test shape: (400, 9)


In [17]:
preprocessor.fit(X_train)
X_train_transformed = preprocessor.transform(X_train)  # New array
X_test_transformed = preprocessor.transform(X_test)    # New array

In [18]:
print("Original X_train shape:", X_train.shape)
print("Transformed X_train shape:", X_train_transformed.shape)
print("Transformed X_test shape:", X_test_transformed.shape)

Original X_train shape: (1600, 9)
Transformed X_train shape: (1600, 9)
Transformed X_test shape: (400, 9)


In [19]:
X_train_transformed_df = pd.DataFrame(X_train_transformed, 
                                      columns=preprocessor.get_feature_names_out())
print(X_train_transformed_df.head())

   Location__Location  Condition__Condition  garage__Garage_Yes  \
0                 3.0                   3.0                 0.0   
1                 0.0                   2.0                 0.0   
2                 3.0                   2.0                 1.0   
3                 1.0                   0.0                 1.0   
4                 1.0                   3.0                 1.0   

   remainder__ID  remainder__Area  remainder__Bedrooms  remainder__Bathrooms  \
0          969.0           4483.0                  4.0                   4.0   
1          241.0           1062.0                  3.0                   3.0   
2          820.0           1422.0                  3.0                   4.0   
3          693.0           2658.0                  2.0                   3.0   
4          421.0           3286.0                  2.0                   4.0   

   remainder__Floors  remainder__YearBuilt  
0                3.0                1933.0  
1                1.0      

In [20]:
lr = LinearRegression()
lr.fit(X_train_transformed,y_train)
y_pred = lr.predict(X_test_transformed)


In [21]:
r2_score(y_test,y_pred )

-0.013357042583030054

In [23]:
for col in X_train_transformed.columns:
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    sns.histplot(X_train_transformed[col],kde=True,)
    plt.title(col)

    plt.subplot(122)
    stats.probplot(X_train_transformed[col],dist="norm",plot=plt)
    plt.title(col)
    plt.show()

AttributeError: 'numpy.ndarray' object has no attribute 'columns'